# Lumis 09: Export the final model for the backend

Converts the final EfficientNet-B0 to **ONNX** so the FastAPI backend runs it with ONNX Runtime instead of TensorFlow
(about 16 MB model, small server image, works on Windows for local development and on Cloud Run).

It also writes `lumis_model_config.json`: classes, temperature, prior adjustment, band thresholds and the scabies cap,
so every setting the test set was scored with travels with the model.

Then it checks, on the **validation** images (the test set stays closed):
1. ONNX gives the same numbers as the Keras model.
2. The backend's own preprocessing (`backend/app/ml/predictor.py`, pasted below unchanged) gives the same pixels as training.
3. End to end, the backend reproduces the validation bands from notebook 04.
4. How long one prediction takes on CPU.

CPU runtime is fine. Takes about 5 minutes.

In [ ]:
RUN = None                        # None = the final model (latest EfficientNet-B0 run)
MODEL_VERSION = 'lumis-effb0-v1'
MODEL_FILE = 'lumis_effb0_v1.onnx'

In [ ]:
!pip install -q tf2onnx onnx onnxruntime simplejpeg
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, json, zipfile, time
import numpy as np, pandas as pd
import tensorflow as tf
import keras

LOCAL = '/content/dermnet'
OUT = f'{BASE}/exports/model'
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    RUN = runs[runs.backbone == 'efficientnetb0'].iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
cal = json.load(open(f'{RUN_DIR}/calibration.json'))
final = json.load(open(f'{BASE}/exports/final_test/metrics.json'))
assert final['run'] == RUN, f"Final test was run on {final['run']}, not {RUN}"
assert final['temperature'] == cal['temperature'] and final['prior_tau'] == cal.get('prior_tau', 0.0), \
    'calibration.json changed after the test was run'

CLASSES, SIZE = cal['classes'], int(info.img_size)
model = keras.models.load_model(info.model_file)
split = pd.read_csv(f'{BASE}/data/lumis_split_{info.split}.csv')
val = split[split.split == 'val'].reset_index(drop=True)
assert sorted(val.lumis_class.unique()) == CLASSES
print(f'Model {RUN} | {len(val)} validation images | TF {tf.__version__}, Keras {keras.__version__}')

## 1. Convert to ONNX

In [ ]:
import tf2onnx, hashlib
onnx_path = f'{OUT}/{MODEL_FILE}'
spec = (tf.TensorSpec((None, SIZE, SIZE, 3), tf.float32, name='image'),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=17, output_path=onnx_path)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

ONNX_SHA = sha256_of(onnx_path)
print(f'Saved {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB), sha256 {ONNX_SHA[:16]}...')

## 2. Write the model config

In [ ]:
o = final['overall']
config = {
    'model_version': MODEL_VERSION,
    'model_file': MODEL_FILE,
    'onnx_sha256': ONNX_SHA,
    'run': RUN,
    'backbone': info.backbone,
    'split': info.split,
    'classes': CLASSES,
    'img_size': SIZE,
    'input': 'float32 (batch, 224, 224, 3), RGB 0-255; resized like tf.image.resize (bilinear, no aspect ratio)',
    'temperature': cal['temperature'],
    'prior_tau': cal.get('prior_tau', 0.0),
    'class_prior': cal.get('class_prior'),
    'logit_adjust': cal.get('logit_adjust', [0.0] * len(CLASSES)),
    'bands': {'high_from': cal['bands']['high_from'], 'refer_below': cal['bands']['refer_below']},
    'cap_at_moderate': ['scabies'] if final.get('scabies_cap') else [],
    'test_results': {'n': final['n_test'], 'evaluated_at': final['evaluated_at'],
                     'accuracy': o['accuracy'][0], 'accuracy_ci95': o['accuracy'][1],
                     'macro_sensitivity': o['macro_sensitivity'][0], 'macro_specificity': o['macro_specificity'][0],
                     'referred': o['referred'][0], 'accuracy_when_answering': o['accuracy_when_answering'][0]},
    'exported_at': time.strftime('%Y-%m-%d %H:%M'),
    'exported_with': {'tensorflow': tf.__version__, 'keras': keras.__version__, 'tf2onnx': tf2onnx.__version__},
}
CONFIG_PATH = f'{OUT}/lumis_model_config.json'
json.dump(config, open(CONFIG_PATH, 'w'), indent=1)
print(json.dumps({k: config[k] for k in ['model_version', 'temperature', 'prior_tau', 'bands', 'cap_at_moderate']}, indent=1))

## 3. Check: ONNX gives the same numbers as Keras

Also compares with the validation logits saved by notebook 03, to confirm this is the same trained model.

In [ ]:
import onnxruntime as ort

def load_like_training(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.cast(tf.image.resize(img, (SIZE, SIZE)), tf.uint8), tf.float32)

paths = [os.path.join(LOCAL, p) for p in val.path]
ds = tf.data.Dataset.from_tensor_slices(paths).map(load_like_training, num_parallel_calls=tf.data.AUTOTUNE).batch(32)
x_train_style = np.concatenate([b.numpy() for b in ds])

keras_logits = model.predict(x_train_style, batch_size=32, verbose=0)
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
onnx_logits = np.concatenate([sess.run(None, {'image': x_train_style[i:i + 32]})[0]
                              for i in range(0, len(x_train_style), 32)])
saved = np.load(f'{RUN_DIR}/val_logits.npy')

checks = {
    'keras_vs_saved_max_diff': float(np.abs(keras_logits - saved).max()),
    'onnx_vs_keras_max_diff': float(np.abs(onnx_logits - keras_logits).max()),
    'onnx_vs_keras_same_class': float((onnx_logits.argmax(1) == keras_logits.argmax(1)).mean()),
}
for k, v in checks.items():
    print(f'{k:28s} {v:.6f}')
assert checks['onnx_vs_keras_same_class'] == 1.0, 'ONNX disagrees with Keras: do not use this export'

## 4. Check: the backend's preprocessing gives the same pixels as training

The next cell is `backend/app/ml/predictor.py`, unchanged. EXIF rotation is switched off for this pixel check
because training did not rotate images; the backend switches it on for phone photos.

In [ ]:
%%writefile predictor.py
"""Lumis skin-condition predictor.

Runs the exported EfficientNet-B0 (ONNX) and turns its output into what the app shows:
a condition, a calibrated confidence and a band (high / moderate / refer).

Preprocessing reproduces training exactly (ml/03_train.ipynb), checked pixel for pixel in ml/09_export.ipynb:
  1. Decode to RGB. JPEG uses libjpeg-turbo with the fast integer DCT, which is what
     tf.io.decode_image uses by default. PNG and WebP are decoded by Pillow.
  2. Resize to 224 x 224 like tf.image.resize: bilinear, half-pixel centers, no antialiasing,
     aspect ratio NOT kept (training stretched every image to a square).
  3. Cast to uint8 (truncation), then to float32 in the 0-255 range. EfficientNet scales and
     normalizes inside the model, so nothing else is done here.
One deliberate difference from training: phone photos are turned upright using their EXIF
orientation tag, because phones often save the pixels sideways.

Output steps, decided in notebooks 04, 04b and 08:
  adjusted logits = logits + logit_adjust     (prior adjustment, training mix, tau 0.75)
  probabilities   = softmax(adjusted logits / temperature)
  band            = high if confidence >= high_from, moderate if >= refer_below, else refer
  scabies (any class in cap_at_moderate) is never shown as high: at most moderate.
"""
from __future__ import annotations

import hashlib
import io
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import simplejpeg
from PIL import Image, UnidentifiedImageError

MAX_PIXELS = 50_000_000          # about 8,000 x 6,000; larger uploads are refused
SUPPORTED_FORMATS = {"JPEG", "PNG", "WEBP"}


class UnsupportedImageError(ValueError):
    """The upload is not an image Lumis can read."""


# ---------------------------------------------------------------- preprocessing

def _apply_exif_orientation(rgb: np.ndarray, orientation: int) -> np.ndarray:
    """Same result as PIL.ImageOps.exif_transpose, on a numpy array."""
    if orientation == 2:
        return rgb[:, ::-1]
    if orientation == 3:
        return rgb[::-1, ::-1]
    if orientation == 4:
        return rgb[::-1]
    if orientation == 5:
        return rgb.transpose(1, 0, 2)
    if orientation == 6:
        return np.rot90(rgb, -1)
    if orientation == 7:
        return rgb.transpose(1, 0, 2)[::-1, ::-1]
    if orientation == 8:
        return np.rot90(rgb, 1)
    return rgb


def decode_image(data: bytes, apply_exif: bool = True) -> np.ndarray:
    """Bytes of a JPEG, PNG or WebP file -> uint8 array of shape (height, width, 3)."""
    try:
        im = Image.open(io.BytesIO(data))
        fmt = im.format
        if fmt not in SUPPORTED_FORMATS:
            raise UnsupportedImageError(f"Unsupported image format: {fmt}. Use JPEG, PNG or WebP.")
        if im.width * im.height > MAX_PIXELS:
            raise UnsupportedImageError("Image is too large.")
        orientation = im.getexif().get(0x0112, 1) if apply_exif else 1
        rgb = None
        if fmt == "JPEG":
            try:
                rgb = simplejpeg.decode_jpeg(data, colorspace="RGB", fastdct=True, fastupsample=False)
            except ValueError:
                rgb = None       # unusual JPEGs (for example CMYK): fall back to Pillow below
        if rgb is None:
            rgb = np.asarray(im.convert("RGB"))
    except UnsupportedImageError:
        raise
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:
        raise UnsupportedImageError("The file could not be read as an image.") from exc
    return _apply_exif_orientation(rgb, orientation)


def resize_like_tf(img: np.ndarray, size: int) -> np.ndarray:
    """tf.image.resize(img, (size, size)) with its defaults: bilinear, half-pixel centers,
    no antialiasing. Returns float32."""
    h, w = img.shape[:2]

    def coords(n_in: int, n_out: int):
        src = (np.arange(n_out, dtype=np.float32) + 0.5) * np.float32(n_in / n_out) - 0.5
        lo = np.floor(src)
        frac = (src - lo).astype(np.float32)
        lo_i = np.clip(lo, 0, n_in - 1).astype(np.int64)
        hi_i = np.clip(lo + 1, 0, n_in - 1).astype(np.int64)
        return lo_i, hi_i, frac

    y0, y1, fy = coords(h, size)
    x0, x1, fx = coords(w, size)
    img = img.astype(np.float32)
    fx3, fy3 = fx[None, :, None], fy[:, None, None]
    top = img[y0][:, x0] * (1 - fx3) + img[y0][:, x1] * fx3
    bottom = img[y1][:, x0] * (1 - fx3) + img[y1][:, x1] * fx3
    return top * (1 - fy3) + bottom * fy3


def preprocess(data: bytes, size: int = 224, apply_exif: bool = True) -> np.ndarray:
    """Image file bytes -> float32 model input of shape (1, size, size, 3), values 0-255."""
    rgb = decode_image(data, apply_exif=apply_exif)
    resized = resize_like_tf(rgb, size).astype(np.uint8)       # truncation, as in training
    return resized.astype(np.float32)[None]


# ---------------------------------------------------------------- model output -> result

@dataclass(frozen=True)
class ModelConfig:
    model_version: str
    classes: list[str]
    img_size: int
    temperature: float
    logit_adjust: list[float]
    high_from: float
    refer_below: float
    cap_at_moderate: list[str] = field(default_factory=list)
    onnx_sha256: str | None = None
    model_file: str = "model.onnx"

    @classmethod
    def load(cls, path: str | Path) -> "ModelConfig":
        c = json.loads(Path(path).read_text())
        return cls(model_version=c["model_version"], classes=list(c["classes"]), img_size=int(c["img_size"]),
                   temperature=float(c["temperature"]), logit_adjust=[float(v) for v in c["logit_adjust"]],
                   high_from=float(c["bands"]["high_from"]), refer_below=float(c["bands"]["refer_below"]),
                   cap_at_moderate=list(c.get("cap_at_moderate", [])), onnx_sha256=c.get("onnx_sha256"),
                   model_file=c.get("model_file", "model.onnx"))


@dataclass(frozen=True)
class Prediction:
    condition: str              # most likely class (also given for 'refer', but the app shows it as inconclusive)
    confidence: float           # calibrated probability of that class, 0-1
    band: str                   # 'high' | 'moderate' | 'refer'
    probabilities: dict[str, float]
    capped: bool                # True if the scabies rule lowered the band from high to moderate
    model_version: str

    def to_dict(self) -> dict:
        return asdict(self)


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def postprocess(logits: np.ndarray, cfg: ModelConfig) -> Prediction:
    """Raw model logits for one image (shape (K,)) -> Prediction."""
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    probs = softmax((logits + np.asarray(cfg.logit_adjust)) / cfg.temperature)
    i = int(probs.argmax())
    conf = float(probs[i])
    condition = cfg.classes[i]
    if conf >= cfg.high_from:
        band = "high"
    elif conf >= cfg.refer_below:
        band = "moderate"
    else:
        band = "refer"
    capped = band == "high" and condition in cfg.cap_at_moderate
    if capped:
        band = "moderate"
    return Prediction(condition=condition, confidence=round(conf, 4), band=band,
                      probabilities={c: round(float(p), 4) for c, p in zip(cfg.classes, probs)},
                      capped=capped, model_version=cfg.model_version)


# ---------------------------------------------------------------- the predictor

def sha256_of(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class LumisPredictor:
    """Load once at startup, then call predict() per upload. Safe to share across requests."""

    def __init__(self, model_path: str | Path, config_path: str | Path, verify_checksum: bool = True):
        import onnxruntime as ort

        self.config = ModelConfig.load(config_path)
        if verify_checksum and self.config.onnx_sha256:
            actual = sha256_of(model_path)
            if actual != self.config.onnx_sha256:
                raise RuntimeError(f"Model file does not match its config (sha256 {actual[:12]}..., "
                                   f"expected {self.config.onnx_sha256[:12]}...). Re-download both from Drive.")
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    @classmethod
    def from_dir(cls, models_dir: str | Path, config_name: str = "lumis_model_config.json") -> "LumisPredictor":
        """Load the config in models_dir and the ONNX file it names."""
        models_dir = Path(models_dir)
        config_path = models_dir / config_name
        return cls(models_dir / ModelConfig.load(config_path).model_file, config_path)

    def logits(self, batch: np.ndarray) -> np.ndarray:
        return self.session.run(None, {self.input_name: batch.astype(np.float32)})[0]

    def predict(self, data: bytes) -> Prediction:
        x = preprocess(data, self.config.img_size)
        return postprocess(self.logits(x)[0], self.config)

In [ ]:
import importlib, predictor
importlib.reload(predictor)
from predictor import LumisPredictor, preprocess, postprocess, ModelConfig
from PIL import Image

x_backend = np.concatenate([preprocess(open(p, 'rb').read(), SIZE, apply_exif=False) for p in paths])
pix_diff = np.abs(x_backend - x_train_style)
rotated = sum(Image.open(p).getexif().get(0x0112, 1) != 1 for p in paths)
checks.update({'pixel_max_diff': float(pix_diff.max()),
               'pixels_differing': float((pix_diff > 0).mean()),
               'val_images_with_exif_rotation': int(rotated)})
print(f"Pixels: max difference {checks['pixel_max_diff']:.0f} (0-255 scale), "
      f"{checks['pixels_differing']:.4%} of values differ")
print('Validation images that carry an EXIF rotation tag:', rotated)

## 5. Check end to end: the backend reproduces notebook 04's validation bands

Loads the backend predictor exactly as the server will (including the sha256 check of the ONNX file).

In [ ]:
pred = LumisPredictor.from_dir(OUT)
cfg = pred.config
backend_logits = np.concatenate([pred.logits(x_backend[i:i + 32]) for i in range(0, len(x_backend), 32)])
results = [postprocess(l, cfg) for l in backend_logits]

# Notebook 04's computation on the saved logits (bands before the scabies cap, as in calibration.json)
z04 = (saved + np.array(cfg.logit_adjust)) / cfg.temperature
probs04 = np.exp(z04 - z04.max(1, keepdims=True))
probs04 /= probs04.sum(1, keepdims=True)
conf04, pred04 = probs04.max(1), probs04.argmax(1)
band04 = np.where(conf04 >= cfg.high_from, 'high', np.where(conf04 >= cfg.refer_below, 'moderate', 'refer'))

y = np.array([CLASSES.index(c) for c in val.lumis_class])
pred_b = np.array([CLASSES.index(r.condition) for r in results])
band_b = np.array(['high' if r.capped else r.band for r in results])     # undo the cap to compare with 04
checks.update({'same_condition_as_04': float((pred_b == pred04).mean()),
               'same_band_as_04': float((band_b == band04).mean()),
               'max_confidence_diff': float(np.abs(np.array([r.confidence for r in results]) - conf04).max())})

rows = []
for b in ['high', 'moderate', 'refer']:
    m = band_b == b
    rows.append({'band': b, 'share_backend': round(m.mean(), 3), 'share_in_calibration_json': cal['bands']['share'][b],
                 'accuracy_backend': round(float((pred_b[m] == y[m]).mean()), 3) if m.any() else None, 'accuracy_in_calibration_json': cal['bands']['accuracy'][b]})
display(pd.DataFrame(rows).set_index('band'))
print(f"Same condition as notebook 04: {checks['same_condition_as_04']:.2%} | same band: {checks['same_band_as_04']:.2%}")
print('Validation accuracy through the backend:', round(float((pred_b == y).mean()), 4))
print('Scabies results lowered from high to moderate by the cap:', sum(r.capped for r in results))

## 6. Speed on CPU (one photo at a time, as the app sends them)

In [ ]:
sample = [open(p, 'rb').read() for p in paths[:30]]
pred.predict(sample[0])                      # warm-up
times = []
for data in sample:
    t0 = time.perf_counter()
    pred.predict(data)
    times.append((time.perf_counter() - t0) * 1000)
checks['ms_per_photo_median'] = round(float(np.median(times)), 1)
print(f"Median {checks['ms_per_photo_median']} ms per photo (decode + resize + model + bands); CP1 NFR1 allows 5 s")

## 7. Save the check report

Then download **both** files from `Drive/Lumis/exports/model/` into `backend/models/` in the repo:
`lumis_effb0_v1.onnx` (stays out of git) and `lumis_model_config.json` (committed).

In [ ]:
json.dump({k: (round(v, 6) if isinstance(v, float) else v) for k, v in checks.items()},
          open(f'{OUT}/export_checks.json', 'w'), indent=1)
print(json.dumps(checks, indent=1))
print('\nFiles in', OUT, ':', sorted(os.listdir(OUT)))